In [1]:
from pathlib import Path
import pandas as pd
import sys
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from scipy.stats import norm

In [6]:
#DATA_PATH = Path("../data/raw/clean_dataset.csv")
DATA_PATH = Path("../../data/raw/dataset_2025-03-01_2026-03-29_external"
                 ".csv")

In [7]:
df = pd.read_csv(DATA_PATH, low_memory=False)
pd.set_option('display.max_columns', None)
#df.info(verbose=True, show_counts=True)

*************

In [4]:
# lead_Ширина

df[~df["lead_Ширина"].isna()]["lead_Ширина"].describe()
df["width_is_anomaly"] = (
    (df["lead_Ширина"] < 10) | (df["lead_Ширина"] > 50)
).astype(int)
df["width_is_anomaly"].value_counts()

width_is_anomaly
0    18775
1      112
Name: count, dtype: int64

In [5]:
def width_category(x):
    if pd.isna(x):
        return "unknown"
    elif x < 10 or x > 50:
        return "anomaly"
    elif x < 25:
        return "small"
    elif x <= 35:
        return "normal"
    else:
        return "large"

df["width_cat"] = df["lead_Ширина"].apply(width_category)

In [6]:
df = df.drop(columns=["lead_Ширина"])

In [7]:
df["width_cat"].value_counts(normalize=True)

width_cat
normal     0.471700
unknown    0.359189
small      0.107852
large      0.055329
anomaly    0.005930
Name: proportion, dtype: float64

***

In [11]:
df[~df["lead_Линейная высота (см)"].isna()]["lead_Линейная высота (см)"].describe()

count    3143.000000
mean       11.618836
std        12.790076
min         2.000000
25%         5.000000
50%         8.000000
75%        13.000000
max       430.000000
Name: lead_Линейная высота (см), dtype: float64

In [14]:
df["lead_height_known"] = df["lead_Линейная высота (см)"].notna().astype(int)
#df = df.drop(columns=["lead_Линейная высота (см)"])

In [16]:
df["lead_height_known"].value_counts()

lead_height_known
0    15744
1     3143
Name: count, dtype: int64

In [ ]:
df = df.drop(columns=["lead_Линейная высота (см)"])

****

In [18]:
df[~df["lead_Вид оплаты"].isna()]["lead_Вид оплаты"].value_counts()

lead_Вид оплаты
Наложенный платеж         18080
Оплата онлайн               601
Оплата на карту             180
Оплата Золотой Короной        4
Name: count, dtype: int64

In [26]:
def payment_type(x):
    if pd.isna(x):
        return "unknown"
    elif x == "Наложенный платеж":
        return "cash_on_delivery"
    else:
        return "online"

df["lead_payment_type"] = df["lead_Вид оплаты"].apply(payment_type)
df = df.drop(columns=["lead_Вид оплаты"])

In [27]:
df.groupby("lead_payment_type")["buyout_flag"].agg(["count", "mean"])

,count,mean
lead_payment_type,,
cash_on_delivery,17175,0.822416
online,769,0.975293
unknown,22,0.318182


*****

In [28]:
# returned_dt -утечка, просто удаляем
df = df.drop(columns=["returned_dt"])

************

In [29]:
df[~df["lead_Служба доставки"].isna()]["lead_Служба доставки"].value_counts()


lead_Служба доставки
СДЭК до ПВЗ      13118
Почта             3217
СДЭК до Двери     2510
Самовывоз           25
Курьер ЕМС           2
Name: count, dtype: int64

In [30]:
def delivery_type(x):
    if pd.isna(x):
        return "unknown"
    elif x == "СДЭК до ПВЗ":
        return "pickup_point"
    elif x in ["СДЭК до Двери", "Курьер ЕМС"]:
        return "door_delivery"
    elif x == "Почта":
        return "post"
    elif x == "Самовывоз":
        return "pickup_point"  # логично объединить
    else:
        return "other"  # на случай мусора

df["lead_delivery_type"] = df["lead_Служба доставки"].apply(delivery_type)
df = df.drop(columns=["lead_Служба доставки"])

In [31]:
df.groupby("lead_delivery_type")["buyout_flag"].agg(["count", "mean"])

,count,mean
lead_delivery_type,,
door_delivery,2418,0.811001
pickup_point,12611,0.847197
post,2922,0.764887
unknown,15,0.133333


*********

In [32]:
df[~df["lead_Компания Отправитель"].isna()]["lead_Компания Отправитель"].value_counts()

lead_Компания Отправитель
ООО ТехПродЗдрав    18581
ООО Здоров            285
Name: count, dtype: int64

In [33]:
df.groupby("lead_Компания Отправитель")["buyout_flag"].agg(["count", "mean"])

,count,mean
lead_Компания Отправитель,,
ООО Здоров,279,0.878136
ООО ТехПродЗдрав,17667,0.828154


In [34]:
# разница мала, и большой дисбаланс значений
df = df.drop(columns=["lead_Компания Отправитель"])

*********

In [35]:
#lead_group_id
df[~df["lead_group_id"].isna()]["lead_group_id"].value_counts()

lead_group_id
700242    6107
546538    6003
0         4257
700246    1817
708650     703
Name: count, dtype: int64

In [36]:
df.groupby("lead_group_id")["buyout_flag"].agg(["count", "mean"])

,count,mean
lead_group_id,,
0,4238,0.848749
546538,5586,0.832975
700242,5683,0.804857
700246,1758,0.818544
708650,701,0.883024


In [39]:
#низкая кардинальность, всего ~5 значений, достаточное количество наблюдений
# есть порядок 0.80 → 0.83 → 0.85 → 0.88

df["lead_group_id"] = df["lead_group_id"].astype("object").fillna("unknown").astype("category")

group_means = df.groupby("lead_group_id")["buyout_flag"].mean()
df["lead_group_quality"] = df["lead_group_id"].map(group_means)
df = df.drop(columns=["lead_group_id"])

*********

In [43]:

df[~df["lead_Масса (гр)"].isna()]["lead_Масса (гр)"].describe()

count     3150.000000
mean       896.606349
std        965.804069
min          1.000000
25%        384.000000
50%        664.000000
75%       1150.000000
max      16950.000000
Name: lead_Масса (гр), dtype: float64

In [44]:
# слишком много пропусков
df["lead_mass_known"] = df["lead_Масса (гр)"].notna().astype(int)

# Ассиметрия, большие выбросы и разбросы
# есть ли масса бинарный сигнал, насколько большая  количественный сигнал
df["lead_mass_log"] = np.log1p(df["lead_Масса (гр)"])

df["lead_mass_log"] = df["lead_mass_log"].fillna(0)
df = df.drop(columns=["lead_Масса (гр)"])

***********

In [46]:
len(df[~df["lead_closed_dt"].isna()]["lead_closed_dt"])


16648

In [48]:
df.groupby(df["lead_closed_dt"].notna())["buyout_flag"].mean()

lead_closed_dt
False    0.462064
True      0.85734
Name: buyout_flag, dtype: object

In [49]:
# buyout → приводит к закрытию лида, утечка, 0.86!
df = df.drop(columns=["lead_closed_dt"])

**********

In [61]:
df[["contact_updated_dt", "contact_created_dt"]]

,contact_updated_dt,contact_created_dt
0,2026-02-14 17:29:40,2025-03-01 03:40:54
1,2026-03-10 12:53:35,2025-03-01 05:30:30
2,2026-02-14 17:29:40,2025-03-01 05:31:15
3,2026-02-14 17:15:47,2025-01-16 19:34:45
4,2026-02-14 17:29:40,2025-03-01 06:01:18
...,...,...
18882,2026-03-29 15:47:58,2026-03-28 06:41:45
18883,2026-03-29 16:02:02,2026-03-29 15:03:41
18884,2026-03-29 16:12:09,2026-01-29 04:10:16
18885,2026-03-29 16:44:16,2026-03-29 16:14:16


In [62]:

df["contact_created_dt"] = pd.to_datetime(df["contact_created_dt"], errors="coerce")
df["contact_updated_dt"] = pd.to_datetime(df["contact_updated_dt"], errors="coerce")

# Потенциально отражает, скорость обработки лида, активность менеджера, интерес клиента
df["contact_update_delay_days"] = (
    df["contact_updated_dt"] - df["contact_created_dt"]
).dt.total_seconds() / (3600 * 24)

df["contact_update_delay_days"].describe()

count    18882.000000
mean       187.934337
std        145.514359
min          0.001551
25%         60.873406
50%        179.920781
75%        289.394303
max        982.040984
Name: contact_update_delay_days, dtype: float64

In [63]:
df["contact_update_delay_days"].count()

np.int64(18882)

In [53]:
df["contact_update_delay_days"] = df["contact_update_delay_days"].fillna(
    df["contact_update_delay_days"].median()
)
df["contact_update_delay_log"] = np.log1p(df["contact_update_delay_days"])
df = df.drop(columns=["contact_created_dt", "contact_updated_dt", "contact_update_delay_days"])

****

In [61]:
df[~df["lead_utm_medium"].isna()]["lead_utm_medium"].value_counts()


lead_utm_medium
cpc                             11975
Неизвестно                        547
Bloger                             26
cpc__rt_view-yes_lead-no_all       15
article_direct                      4
cpm                                 2
organic                             1
Name: count, dtype: int64

In [65]:
# Смотрим платный трафик или бесплатный CPC = Cost Per Click и CPM = Cost Per Mille (1000 показов). Это индикатор платного привлечения клиента, тут еще можно поэкспериментировать
# если время будет
df["is_paid_traffic"] = df["lead_utm_medium"].astype(str).str.lower().str.contains("cpc|cpm").astype(int)
df = df.drop(columns=["lead_utm_medium"])

In [63]:
df.groupby("is_paid_traffic")["buyout_flag"].mean()

is_paid_traffic
0    0.879922
1    0.798002
Name: buyout_flag, dtype: object

********

In [66]:
df[~df["lead_Категория и варианты выбора"].isna()]["lead_Категория и варианты выбора"].value_counts()


lead_Категория и варианты выбора
S                5602
I                2324
D                1367
C                 356
Нет категории     190
Name: count, dtype: int64

In [68]:
df.groupby("lead_Категория и варианты выбора")["buyout_flag"].mean()

lead_Категория и варианты выбора
C                0.794393
D                0.823973
I                0.778654
S                0.817967
Нет категории    0.790323
Name: buyout_flag, dtype: object

In [71]:
# отражает “массовость категории”
df["lead_Категория и варианты выбора"] = df["lead_Категория и варианты выбора"].fillna("unknown").replace({"Нет категории": "unknown"})
freq = df["lead_Категория и варианты выбора"].value_counts(normalize=True)
df["lead_category_freq"] = df["lead_Категория и варианты выбора"].map(freq)

In [ ]:
df = df.drop(columns=["lead_Категория и варианты выбора"])

In [73]:
df["lead_category_freq"].value_counts()

lead_category_freq
0.489120    9238
0.296606    5602
0.123048    2324
0.072378    1367
0.018849     356
Name: count, dtype: int64

************

In [74]:
# received_dt -утечка, просто удаляем, факт завершённой сделки
df = df.drop(columns=["received_dt"])

In [77]:
df[~df["lead_Модель телефона"].isna()]["lead_Модель телефона"].value_counts()


lead_Модель телефона
Смартфон             7924
Не удалось узнать    5739
Кнопочный             272
Name: count, dtype: int64

In [80]:
df.groupby("lead_Модель телефона")["buyout_flag"].mean()
#df["lead_Модель телефона"].isna().mean()

lead_Модель телефона
Кнопочный             0.76834
Не удалось узнать     0.81006
Смартфон             0.821717
Name: buyout_flag, dtype: object

In [64]:
df["lead_Модель телефона"] = df["lead_Модель телефона"].fillna("unknown")
df["is_feature_phone"] = (df["lead_Модель телефона"] == "Кнопочный").astype(int)
# todo сделать target encoding


In [ ]:
df = df.drop(columns=["lead_Модель телефона"])

In [66]:
df.groupby("is_feature_phone")["buyout_flag"].mean()

is_feature_phone
0    0.82922
1    0.76834
Name: buyout_flag, dtype: object

*********

In [72]:
col1 = "lead_Дата перехода Передан в доставку"
df[col1] = pd.to_datetime(df[col1], unit="s", utc=True)
df[col1] = df[col1].dt.date
df["handed_to_delivery_dt"] = pd.to_datetime(df["handed_to_delivery_dt"], errors='coerce')
df["handed_to_delivery_dt_modif"] = df["handed_to_delivery_dt"].dt.date

df['date1'] = df["handed_to_delivery_dt_modif"]
df['date2'] = df[col1]

# Считаем совпадения
matches = df['date1'] == df['date2']
match_count = matches.sum()
total_count = len(df)

print(f"Совпадает: {match_count} ({match_count/total_count*100:.2f}%)")
print(f"Не совпадает: {(~matches).sum()} ({(~matches).sum()/total_count*100:.2f}%)")
nn = len(df[~df[col1].isna()])
print("Процент совпадения не nan значений", nn, match_count/nn * 100, "%")

Совпадает: 10 (0.05%)
Не совпадает: 18877 (99.95%)
Процент совпадения не nan значений 5885 0.16992353440951571 %


In [73]:
# Итого признак "lead_Дата перехода Передан в доставку" разреженный
# дубль handed_to_delivery_dt, удаляем
df = df.drop(columns=["lead_Дата перехода Передан в доставку", "handed_to_delivery_dt_modif"])

**********

In [74]:
# Отмечаем, заказ дошел/не дошёл до доставки
df["handed_to_delivery_missing"] = df["handed_to_delivery_dt"].isna().astype(int)

In [75]:
# Производные признаки
#Время до передачи в доставку
df["lead_created_dt"] = pd.to_datetime(df["lead_created_dt"], errors='coerce')
df["lead_to_delivery_days"] = (
    df["handed_to_delivery_dt"] - df["lead_created_dt"]
).dt.total_seconds() / 86400

In [24]:
#Время от продажи до доставки
df["sale_date_dt"] = pd.to_datetime(df["sale_date_dt"], errors='coerce')
df["sale_to_delivery_days"] = (
    df["handed_to_delivery_dt"] - df["sale_date_dt"]
).dt.total_seconds() / 86400

In [26]:
df["sale_to_delivery_days"].describe()

count    18241.000000
mean         2.295089
std          1.846105
min          0.273299
25%          1.360382
50%          1.575336
75%          2.638981
max         68.492546
Name: sale_to_delivery_days, dtype: float64

In [30]:
df["sale_to_delivery_log"] = np.log1p(df["sale_to_delivery_days"])
df["sale_to_delivery_log"].describe()

# Корректное заполнение пропусков, -1 нет события
df["sale_to_delivery_missing"] = df["sale_to_delivery_log"].isna().astype(int)
df["sale_to_delivery_log"] = df["sale_to_delivery_log"].fillna(-1)


In [33]:
df["lead_to_delivery_days_log"] = np.log1p(df["lead_to_delivery_days"])
df["lead_to_delivery_days_log"].describe()

# Корректное заполнение пропусков, -1 нет события
df["lead_to_delivery_days_missing"] = df["lead_to_delivery_days_log"].isna().astype(int)
df["lead_to_delivery_days_log"] = df["lead_to_delivery_days_log"].fillna(-1)
df["lead_to_delivery_days_log"].describe()

count    18887.000000
mean         1.118831
std          0.904190
min         -1.000000
25%          0.679839
50%          0.889024
75%          1.413233
max          6.535293
Name: lead_to_delivery_days_log, dtype: float64

In [36]:
#!!!!! sale_date_dt, lead_created_dt

df = df.drop(columns=["lead_to_delivery_days", "sale_to_delivery_days",
                      "handed_to_delivery_dt"])

******

In [76]:
# ID рекламной кампании
df[~df["lead_utm_campaign"].isna()]["lead_utm_campaign"].value_counts()

lead_utm_campaign
116543546     685
115509254     560
Неизвестно    547
118448375     376
700615408     339
             ... 
707281666       1
707338875       1
707363652       1
707177354       1
707306601       1
Name: count, Length: 1223, dtype: int64

In [42]:
# Уберем мусор
df["lead_utm_campaign"] = (df["lead_utm_campaign"].replace
                           (["{campaing_id}", "Неизвестно"], np.nan))

# Признак сильно разреженный, отмечаем где не заполнен
df["lead_utm_campaign_missing"] = df["lead_utm_campaign"].isna().astype(int)

# В теории можно Target Encoding, но пока простейший вариант, хвост
# сократить
top_k = 20  # можно 20–50
top_values = df["lead_utm_campaign"].value_counts().head(top_k).index

df["lead_utm_campaign_grouped"] = df["lead_utm_campaign"].where(
    df["lead_utm_campaign"].isin(top_values),
    "other"
)

In [43]:
df["lead_utm_campaign_grouped"].value_counts()

lead_utm_campaign_grouped
other        14583
116543546      685
115509254      560
118448375      376
700615408      339
115367108      272
702829935      215
118075886      197
118108441      170
702103466      169
119815667      153
114317039      139
118751984      138
114252495      133
702279529      133
119707922      130
701745533      110
702467942      100
702013262       96
702239869       96
114824436       93
Name: count, dtype: int64

In [44]:
df = df.drop(columns=["lead_utm_campaign"])

*********

In [48]:


df[~df["contact_Источник трафика"].isna()]["contact_Источник трафика"].value_counts()

contact_Источник трафика
dzen.ru                          1047
Yandex Direct                     799
m.dzen.ru                         350
Yandex                            145
artraid.promo.page                144
                                 ... 
punkt-a.info                        1
allcafe.ru                          1
93.ru                               1
uznayvse.ru                         1
Переходы по ссылкам на сайтах       1
Name: count, Length: 469, dtype: int64

In [50]:
df["traffic_source_missing"] = df["contact_Источник трафика"].isna().astype(int)

top_k = 20

top_values = (
    df["contact_Источник трафика"]
    .value_counts()
    .head(top_k)
    .index
)

df["traffic_source_grouped"] = df["contact_Источник трафика"].where(
    df["contact_Источник трафика"].isin(top_values),
    "other"
)

In [52]:
df["traffic_source_grouped"].value_counts()

traffic_source_grouped
other                  15718
dzen.ru                 1047
Yandex Direct            799
m.dzen.ru                350
Yandex                   145
artraid.promo.page       144
Прямой заход              91
Google                    90
ria.ru                    67
gismeteo.ru               60
m.ura.news                47
лечу-варикоз.рф           45
rlsnet.ru                 44
pogoda7.ru                40
mail.ru                   35
tv.yandex.ru              31
pravda.ru                 28
vidal.ru                  28
progoroduhta.ru           28
Переходы по рекламе       25
otzovik.com               25
Name: count, dtype: int64

In [53]:
df = df.drop(columns=["contact_Источник трафика"])

*************

In [87]:
df[[
    "lead_created_dt",
    "lead_Дата перехода в Сборку",
    "handed_to_delivery_dt",
    "issued_or_pvz_dt"
]].dropna().head()

,lead_created_dt,lead_Дата перехода в Сборку,handed_to_delivery_dt,issued_or_pvz_dt
9663,2025-09-30 03:35:42,2025-09-29 00:00:00+00:00,2025-10-02 10:09:44,2025-10-07 05:46:02
9664,2025-09-23 21:04:52,2025-09-29 00:00:00+00:00,2025-10-01 11:57:46,2025-10-22 03:18:50
9665,2025-09-29 09:23:21,2025-09-29 00:00:00+00:00,2025-10-01 07:16:53,2025-10-06 11:02:05
9666,2025-09-30 02:24:41,2025-09-29 00:00:00+00:00,2025-10-01 07:30:13,2025-10-07 14:18:56
9667,2025-09-30 03:32:29,2025-09-29 00:00:00+00:00,2025-10-01 07:28:08,2025-10-07 09:01:59


In [7]:
# --- 1. приведение к datetime + убираем tz ---
df["lead_created_dt"] = pd.to_datetime(df["lead_created_dt"], errors="coerce").dt.tz_localize(None)

df["lead_Дата перехода в Сборку"] = pd.to_datetime(
    df["lead_Дата перехода в Сборку"], errors="coerce"
).dt.tz_localize(None)

In [18]:
start = pd.Timestamp("2025-03-01")
end   = pd.Timestamp("2026-03-29")

date_cols = [
    "lead_created_dt",
    "lead_Дата перехода в Сборку"
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

    df.loc[
        (df[col] < start) | (df[col] > end),
        col
    ] = pd.NaT

df["lead_to_assembly_days"] = (
    df["lead_Дата перехода в Сборку"] - df["lead_created_dt"]
).dt.total_seconds() / 86400

In [19]:
# Ловушка, ибо получилось тем больше разница тем вероятнее выкуп
df[df["lead_to_assembly_days"] > 30]["buyout_flag"].mean()

np.float64(0.928)

In [20]:
# --- 2. delay (сборка - создание) ---
df["lead_to_assembly_days"] = (
    df["lead_Дата перехода в Сборку"] - df["lead_created_dt"]
).dt.total_seconds() / 86400

# --- 3. исправление timezone-сдвига (+1 день) ---
df["lead_to_assembly_days"] = df["lead_to_assembly_days"] + 1

# --- 4. флаг события ---
df["assembly_flag"] = df["lead_Дата перехода в Сборку"].notna().astype(int)

# --- 5. контроль хвоста ---
df["lead_to_assembly_long"] = (df["lead_to_assembly_days"] > 30).astype(int)

df["lead_to_assembly_days"] = df["lead_to_assembly_days"].clip(upper=30)

# --- 6. обработка пропусков ---
df["lead_to_assembly_days"] = df["lead_to_assembly_days"].fillna(-1)

# --- 7. удаление исходного признака ---
df = df.drop(columns=["lead_Дата перехода в Сборку"])

*************************

In [21]:
df[~df["lead_utm_term"].isna()]["lead_utm_term"].value_counts()



lead_utm_term
dem_otek_clip1_otek                       1665
dem_varicose_clip1_varikoz                1057
yur_sustavy_applicator_landing-artraid     855
dem_sustav_clip002                         799
yur_sleep_maska_sleep                      729
                                          ... 
dem2_varicose_Confeti_varikoz                1
999999                                       1
sustavy_clip                                 1
catalog                                      1
nim_varicose_bandage_varikoz_clip_1          1
Name: count, Length: 162, dtype: int64

In [29]:
df["lead_utm_term"] = df["lead_utm_term"].replace(
    ["Неизвестно"],
    np.nan
)

df["utm_term_missing"] = df["lead_utm_term"].isna().astype(int)

# --- 3. top-K + other ---
top_k = 40
top_values = df["lead_utm_term"].value_counts().head(top_k).index

df["utm_term_grouped"] = df["lead_utm_term"].where(
    df["lead_utm_term"].isin(top_values),
    "other"
)
# Можно было поиграться с frequency encoding, но это уже тюнинг

In [ ]:
df = df.drop(columns=["lead_utm_term"])

In [30]:
df["utm_term_grouped"].value_counts()

utm_term_grouped
other                                              7106
dem_otek_clip1_otek                                1665
dem_varicose_clip1_varikoz                         1057
yur_sustavy_applicator_landing-artraid              855
dem_sustav_clip002                                  799
yur_sleep_maska_sleep                               729
yur_varicose_bandage_landing-artraid-2              714
dem_sleep_clip                                      650
yur_sustavy_applicator_poyasnica                    521
but_varicose_bandage_varikoz_clip                   393
nim_varicose_bandage_varikoz_clip                   368
nim_varicose_bandage_landing-artraid-2              347
yur_varicose_bandage_varikoz_clip                   345
nim_sleep_maska_sleep                               315
dem_davlenie2                                       190
dem_sustav_clip                                     185
dem_ssz                                             178
but_varicose_bandage_landing-ar

*********************

In [33]:
len(df[~df["lead_utm_sky"].isna()]["lead_utm_sky"])

11492

In [32]:
df[~df["lead_utm_sky"].isna()]["lead_utm_sky"].value_counts().head(30)


lead_utm_sky
---autotargeting                             9467
{keyword}                                      90
p=but                                          80
artraid                                        45
длительное нарушение сна                       29
нарушение сна человека                         25
артрейд                                        14
варикоз                                        14
нарушение сна раздражительность                13
artraid микросферы                             12
лечение варикоза                               10
флеболог и узи вен                             10
варикоз лазерная                                9
artraid официальный сайт                        9
undefined                                       9
флеболог лечит варикоз                          9
прием флеболога                                 9
микросферы артрейд                              8
варикоз на ногах у мужчин                       8
артраид микросферы купить            

In [34]:
df["lead_utm_sky"] = df["lead_utm_sky"].replace(
    ["{keyword}"],
    np.nan
)

# Базовый missing-флаг
df["utm_sky_missing"] = df["lead_utm_sky"].isna().astype(int)

# отдельный тип трафика  часто сильно влияет
df["utm_sky_autotarget"] = (df["lead_utm_sky"] == "---autotargeting").astype(int)

# выделим бренд
df["utm_sky_brand"] = df["lead_utm_sky"].str.contains(
    "artraid|артрейд",
    case=False,
    na=False
).astype(int)

# семантика запроса
df["utm_sky_varicose"] = df["lead_utm_sky"].str.contains(
    "варикоз|вены|флеболог",
    case=False,
    na=False
).astype(int)

df["utm_sky_sleep"] = df["lead_utm_sky"].str.contains(
    "сон|sleep",
    case=False,
    na=False
).astype(int)

In [38]:
df["utm_sky_autotarget"].value_counts()

utm_sky_autotarget
1    9467
0    9420
Name: count, dtype: int64

In [ ]:
df = df.drop(columns=["lead_utm_sky"])

************************

In [39]:
# rejected_dt - утечка, просто удаляем
df = df.drop(columns=["rejected_dt"])

******************

In [42]:
df[~df["lead_group"].isna()]["lead_group"].value_counts()


lead_group
yur     4674
but     1211
gus       19
inr        7
imp        3
kru        2
nil        2
nim        2
dem2       2
mea        1
dem        1
Name: count, dtype: int64

In [44]:
# флаг пропуска
df["lead_group_missing"] = df["lead_group"].isna().astype(int)

# топ-2 группы + other
main_groups = ["yur", "but"]

df["lead_group_grouped"] = df["lead_group"].where(
    df["lead_group"].isin(main_groups),
    "other"
)

# удаление исходного столбца


In [45]:
df["lead_group_grouped"].value_counts()

lead_group_grouped
other    13002
yur       4674
but       1211
Name: count, dtype: int64

In [ ]:
df = df.drop(columns=["lead_group"])

***************

In [50]:
len(df[~df["lead_Проблема"].isna()]["lead_Проблема"])



18860

In [48]:
# missing-флаг
df["problem_missing"] = df["lead_Проблема"].isna().astype(int)

# --- 3. укрупнение категорий ---
main_groups = [
    "Суставы и позвоночник",
    "Варикоз",
    "Сердечно-сосудистые заболевания",
    "Бессоница",
    "Головные боли",
    "Отеки",
    "Зрительная система",
    "Давление",
    "Инсульт",
    "Боли и тяжесть в ногах"
]

df["problem_grouped"] = df["lead_Проблема"].where(
    df["lead_Проблема"].isin(main_groups),
    "other"
)



In [49]:
df["problem_grouped"].value_counts()

problem_grouped
Суставы и позвоночник              8347
Варикоз                            5403
Сердечно-сосудистые заболевания    1238
Бессоница                          1180
Головные боли                       643
other                               460
Отеки                               459
Зрительная система                  382
Давление                            303
Инсульт                             240
Боли и тяжесть в ногах              232
Name: count, dtype: int64

In [ ]:
df = df.drop(columns=["lead_Проблема"])